# Polly HF SFT Training (Colab Pro+)

Runs scripts/train/hf_train_polly.py against issdandavis/polly-training-data and pushes the LoRA adapter to issdandavis/polly-1.

**Before running:** add your Hugging Face write-token to Colab Secrets as HF_TOKEN (left sidebar -> key icon -> + Add new secret). The notebook never sees the raw token in code.

In [ ]:
# 1. Clone SCBE-AETHERMOORE training branch and install deps
!git clone --depth 1 -b neurogolf/ant-colony-solvers https://github.com/issdandavis/SCBE-AETHERMOORE.git scbe
%cd scbe
!pip -q install -U "transformers>=4.45" "trl>=0.12" "peft>=0.13" "accelerate>=1.0" "bitsandbytes>=0.44" "datasets>=3.0" "huggingface_hub>=0.25"

In [ ]:
# 2. Pull HF_TOKEN from Colab Secrets (preferred) or prompt
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab secrets')
except Exception as e:
    from getpass import getpass
    os.environ['HF_TOKEN'] = getpass('Paste HF_TOKEN (write scope): ')
assert os.environ.get('HF_TOKEN'), 'HF_TOKEN not set'

In [ ]:
# 3. Sanity check the GPU and dataset before training
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
from datasets import load_dataset
ds = load_dataset('issdandavis/polly-training-data', split='train', token=os.environ['HF_TOKEN'])
print('rows:', len(ds), '| columns:', ds.column_names)
print('first row roles:', [m['role'] for m in ds[0]['messages']])

In [ ]:
# 4. Train and push to issdandavis/polly-1
# T4  (15 GB):  --batch-size 2 --grad-accum 8
# L4  (22 GB):  --batch-size 4 --grad-accum 4 (default)
# A100 (40+):   --batch-size 8 --grad-accum 2 --no-quant
!python scripts/train/hf_train_polly.py \
    --base-model Qwen/Qwen2.5-0.5B \
    --dataset issdandavis/polly-training-data \
    --output-repo issdandavis/polly-1 \
    --output-dir ./out/polly-lora \
    --epochs 3 \
    --batch-size 4 \
    --grad-accum 4 \
    --lr 2e-4 \
    --max-seq 1024 \
    --max-grad-norm 1.0 \
    --push

In [ ]:
# 5. Smoke check the pushed adapter
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
info = api.repo_info('issdandavis/polly-1', repo_type='model')
print('repo:', info.id, '| sha:', info.sha[:8], '| files:', len(info.siblings))
for s in info.siblings[:20]:
    print(' -', s.rfilename)